# Single-class object of interest — YOLO11s-seg v2 (BCE + Dice)

Same recipe as `train_rtdetr_l_v2.ipynb`. Only the model size changes: **YOLO11s-seg** instead of YOLO11l-seg.

| Setting | L v2 | **S v2** |
|---|---|---|
| model | `yolo11l-seg.pt` | **`yolo11s-seg.pt`** |
| mask loss | 0.5 BCE + 0.5 Dice | **same** |
| `cls` | 0.4 | **same** |
| `mosaic` / `close_mosaic` | 0.4 / 10 | **same** |
| `cos_lr` | True | **same** |
| `epochs` / `batch` / `imgsz` | 200 / 4 / 640 | **same** |
| `patience` | 20 | **same** |

Same single-class `train_v2/dataset` (`nc: 1`, class `0` / `object`). Do **not** run the download from this notebook. Weights go to `train_v2/runs/v2/yolo11s_seg_object_bce_dice/` — the L v2 folder is not touched.

## 0. Download the dataset (run this in a terminal first)

Skip this if `train_v2/dataset` already exists from the L runs.

From the **repo root** in PowerShell:

```powershell
python train_v2/utils/download_dataset.py
```

## 1. Setup and environment check

In [1]:
from pathlib import Path
import os
import sys

import torch
import ultralytics
from ultralytics import YOLO
from ultralytics.utils import SETTINGS

SETTINGS["tensorboard"] = False

HERE = Path.cwd().resolve()
if not (HERE / "utils" / "download_dataset.py").exists():
    HERE = HERE / "train_v2"

sys.path.insert(0, str(HERE / "utils"))
from augment import LABEL_AWARE_AUG
from dice_loss import apply_bce_dice_mask_loss

DATA_YAML = HERE / "dataset" / "data.yaml"
RUNS = HERE / "runs"
L_V2_DIR = RUNS / "v2" / "yolo11l_seg_object_bce_dice"
PROJECT = RUNS / "v2"
RUN_NAME = "yolo11s_seg_object_bce_dice"
V2_DIR = PROJECT / RUN_NAME

os.environ["MLFLOW_EXPERIMENT_NAME"] = "train_v2_yolo11s_seg_bce_dice"
os.environ["MLFLOW_RUN"] = RUN_NAME

if V2_DIR.resolve() == L_V2_DIR.resolve():
    raise RuntimeError("S v2 save path collided with the L v2 run.")
if V2_DIR.exists():
    print(
        f"NOTE: {V2_DIR} already exists. exist_ok=False so Ultralytics "
        "will create a new folder (…2) instead of overwriting."
    )

TRAIN_AUG = dict(LABEL_AWARE_AUG)
TRAIN_AUG["mosaic"] = 0.4
TRAIN_AUG["close_mosaic"] = 10

print("=" * 70)
print("TRAIN_V2 — YOLO11s-seg v2 (BCE+Dice)")
print("=" * 70)
print(f"Ultralytics : {ultralytics.__version__}")
print(f"PyTorch     : {torch.__version__}")
print(f"CUDA        : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU         : {torch.cuda.get_device_name(0)}")
print()
print(f"train_v2    : {HERE}")
print(f"data.yaml   : {DATA_YAML}")
print(f"yaml exists : {DATA_YAML.exists()}")
print(f"L v2 dir    : {L_V2_DIR}  exists={L_V2_DIR.exists()}")
print(f"S project   : {PROJECT}")
print(f"S run name  : {RUN_NAME}")
print(f"S save dir  : {V2_DIR}")
print(f"MLflow exp  : {os.environ['MLFLOW_EXPERIMENT_NAME']}")
print(f"MLflow run  : {os.environ['MLFLOW_RUN']}")
if not DATA_YAML.exists():
    raise FileNotFoundError(
        "Dataset not found. From the repo root run:\n"
        "  python train_v2/utils/download_dataset.py"
    )
print("=" * 70)

c:\Users\Haqkiem\AppData\Local\Programs\Python\Python310\lib\site-packages\torch\cuda\__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


TRAIN_V2 — YOLO11s-seg v2 (BCE+Dice)
Ultralytics : 8.4.146
PyTorch     : 2.11.0+cu128
CUDA        : True
GPU         : NVIDIA GeForce RTX 4050 Laptop GPU

train_v2    : C:\Users\Haqkiem\OneDrive\UNIKL\July-2026\Competition\AIIC\PETROSAINS\train_v2
data.yaml   : C:\Users\Haqkiem\OneDrive\UNIKL\July-2026\Competition\AIIC\PETROSAINS\train_v2\dataset\data.yaml
yaml exists : True
L v2 dir    : C:\Users\Haqkiem\OneDrive\UNIKL\July-2026\Competition\AIIC\PETROSAINS\train_v2\runs\v2\yolo11l_seg_object_bce_dice  exists=True
S project   : C:\Users\Haqkiem\OneDrive\UNIKL\July-2026\Competition\AIIC\PETROSAINS\train_v2\runs\v2
S run name  : yolo11s_seg_object_bce_dice
S save dir  : C:\Users\Haqkiem\OneDrive\UNIKL\July-2026\Competition\AIIC\PETROSAINS\train_v2\runs\v2\yolo11s_seg_object_bce_dice
MLflow exp  : train_v2_yolo11s_seg_bce_dice
MLflow run  : yolo11s_seg_object_bce_dice


## 2. Confirm the local set is single-class

Every polygon should already be class `0` after the download script. This cell only checks; it does not rewrite labels.

In [ ]:
import yaml

cfg = yaml.safe_load(DATA_YAML.read_text(encoding="utf-8"))
print("nc   :", cfg.get("nc"))
print("names:", cfg.get("names"))

class_ids = set()
n_objects = 0
label_files = list((HERE / "dataset" / "labels").rglob("*.txt"))
for path in label_files:
    for line in path.read_text(encoding="utf-8", errors="ignore").splitlines():
        parts = line.split()
        if len(parts) < 5:
            continue
        class_ids.add(int(float(parts[0])))
        n_objects += 1

print(f"label files : {len(label_files)}")
print(f"objects     : {n_objects}")
print(f"class ids   : {sorted(class_ids)}")
if class_ids != {0}:
    raise ValueError(f"Expected only class 0, found {sorted(class_ids)}")
print("OK — single class `object` (id 0). SKU labels are ignored.")

nc   : 1
names: {0: 'object'}


## 3. Switch mask loss to BCE + Dice

Patches `v8SegmentationLoss.single_mask_loss` to **0.5 BCE + 0.5 Dice** (same hybrid as L v2). Box, classification, and DFL stay Ultralytics defaults.

In [ ]:
apply_bce_dice_mask_loss()

## 4. Augmentation (label-aware + light mosaic)

Keeps the v1 photometric / small-geometry augs, then turns mosaic back on at `0.4` and disables it for the last 10 epochs (`close_mosaic=10`).

In [ ]:
print("v2 train() aug kwargs:")
for key, value in TRAIN_AUG.items():
    print(f"  {key:14s} {value}")

## 5. Train YOLO11s-seg (BCE + Dice, 200 epochs)

Weights land in `train_v2/runs/v2/yolo11s_seg_object_bce_dice/` (not the L v2 folder).

`batch=4` and `imgsz=640` are unchanged. S uses less VRAM than L.

In [ ]:
last = V2_DIR / "weights" / "last.pt"
resume = last.exists()
seg_model = YOLO(str(last) if resume else "yolo11s-seg.pt")
print(f"resume={resume}  weights={last if resume else 'yolo11s-seg.pt'}")

seg_results = seg_model.train(
    data=str(DATA_YAML),
    epochs=200,
    imgsz=640,
    batch=4,
    workers=0,
    patience=20,
    device=0 if torch.cuda.is_available() else "cpu",
    project=str(PROJECT),
    name=RUN_NAME,
    exist_ok=resume,
    resume=resume,
    cls=0.4,
    cos_lr=True,
    **TRAIN_AUG,
)

print(seg_results)

## 6. Quick sanity check on a val image

Loads `best.pt` from this S v2 segmentation run.

In [ ]:
best = V2_DIR / "weights" / "best.pt"
if not best.exists():
    raise FileNotFoundError(f"No weights yet: {best}")

val_images = sorted(
    p for p in (HERE / "dataset" / "images" / "val").iterdir()
    if p.suffix.lower() in {".jpg", ".jpeg", ".png"}
)
sample = val_images[0]
print("Sample:", sample)

pred_model = YOLO(str(best))
results = pred_model.predict(source=str(sample), conf=0.25, verbose=False)[0]
print(f"detections: {len(results.boxes)}  (class ignored — all are `object`)")
out = HERE / "preview_seg_s_v2.jpg"
results.save(filename=str(out))
print("Wrote", out)